# Currency & Economy

Currency flow simulation for Animal Company. Models instance value-in/out, wallet balances, Key Card progression, and Battle Pass economics.

> Reference: `animalco_economy.md` in brain-rag vault

In [1]:
from IPython.display import HTML
display(HTML('''
<style id="aco-toggle-style"></style>
<button id="aco-toggle-btn"
  onclick="
    var style = document.getElementById('aco-toggle-style');
    if (style.innerHTML === '') {
      style.innerHTML = '.jp-Cell-inputWrapper { display: none !important; }';
      this.innerHTML = 'Show Code';
    } else {
      style.innerHTML = '';
      this.innerHTML = 'Hide Code';
    }
  "
  style="padding: 6px 16px; font-size: 13px; cursor: pointer;
         border: 1px solid #ccc; border-radius: 4px; background: #f5f5f5;">
  Hide Code
</button>
'''))

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

from aco_model.models import EconomyParams, RetentionCurve
from aco_model.retention import load_installs, simulate
from aco_model.economy import simulate_economy
from aco_model.config import load_config
from aco_model.state import load_state, save_state, DEFAULT_STATE_PATH

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

_HEADLESS = os.environ.get('ACO_HEADLESS') == '1'

cfg = load_config()
state = load_state()

if state is not None:
    installs = load_installs(cfg.installs_path)
    curve = RetentionCurve(anchors=state.retention_anchors)
    sim = simulate(installs, curve, state.sim_days)
    econ_params = state.economy or cfg.economy
    print(f'=== Loaded from shared state ===')
else:
    installs = load_installs(cfg.installs_path)
    sim = simulate(installs, cfg.retention, cfg.sim_days)
    econ_params = cfg.economy
    print(f'=== Using config.yaml defaults ===')

print(f'  DAU range: {sim.dau.min():,} – {sim.dau.max():,}')
# Load extra economy values from raw config (not in Pydantic model)
import yaml as _yaml
with open('config.yaml') as _f:
    _raw_cfg = _yaml.safe_load(_f) or {}
_econ_raw = _raw_cfg.get('economy', {})
_saved_bronze_cost = _econ_raw.get('bronze_coin_cost', 10.00)
_saved_nut_usd = _econ_raw.get('nut_value_usd', 0.01)
_saved_scrap_usd = _econ_raw.get('scrap_value_usd', 0.005)
print(f'  Instances/day: {econ_params.instances_per_day}')

=== Loaded from shared state ===
  DAU range: 15,182 – 61,725
  Instances/day: 1


## 1. Resource Values & Exchange Rates

In [3]:
# ── Resource value inputs (USD per unit) ──────────────────────────────────
coin_value = widgets.FloatText(value=econ_params.coin_to_usd, description='1 Coin = $', step=0.1,
                                layout=widgets.Layout(width='180px'))
nut_value = widgets.FloatText(value=_saved_nut_usd, description='1 Nut = $', step=0.001,
                               layout=widgets.Layout(width='180px'))
scrap_value = widgets.FloatText(value=_saved_scrap_usd, description='1 Scrap = $', step=0.001,
                                 layout=widgets.Layout(width='180px'))

# ── Key Card tier inputs ──────────────────────────────────────────────────
_kc_tiers = econ_params.keycard_tiers
_kc_names = [kc.name for kc in _kc_tiers]

_w = widgets.Layout(width='120px')
_w_ro = widgets.Layout(width='120px')

# Bronze buy price is the only settable coin cost — others cascade
bronze_coin_cost = widgets.FloatText(value=_saved_bronze_cost, step=0.01, layout=_w)

# Derived buy prices (read-only) for higher tiers
_kc_derived_prices = {}
_kc_usd_labels = {}
for kc in _kc_tiers:
    _kc_derived_prices[kc.name] = widgets.HTML('')
    _kc_usd_labels[kc.name] = widgets.HTML('')

_kc_merge_inputs = {}
_kc_cards_inputs = {}
for kc in _kc_tiers:
    _kc_merge_inputs[kc.name] = widgets.FloatText(value=kc.merge_cost_nuts, step=10, layout=_w)
    _kc_cards_inputs[kc.name] = widgets.IntText(value=kc.cards_required, step=1, layout=_w)

# Bronze is base tier — disable merge inputs
_kc_merge_inputs[_kc_names[0]].disabled = True
_kc_cards_inputs[_kc_names[0]].disabled = True


# ── Instance Loot inputs (per-tier avg output) ───────────────────────────
_loot_nuts = {}
_loot_scrap = {}
_loot_coins = {}
_loot_gear = {}
_loot_kc_drop = {}
_lw = widgets.Layout(width='100px')
for tier in econ_params.instance_tiers:
    _loot_nuts[tier.name] = widgets.FloatText(value=tier.nuts_earned, step=5, layout=_lw)
    _loot_scrap[tier.name] = widgets.FloatText(value=tier.scrap_earned, step=10, layout=_lw)
    _loot_coins[tier.name] = widgets.FloatText(value=tier.coins_earned, step=0.5, layout=_lw)
    _loot_gear[tier.name] = widgets.FloatText(value=tier.gear_value_usd, step=0.05, layout=_lw)
    _loot_kc_drop[tier.name] = widgets.FloatText(value=tier.keycard_drop_chance * 100, step=1, layout=_lw)

# ── Economy sliders ───────────────────────────────────────────────────────
runs_slider = widgets.IntSlider(value=econ_params.instances_per_day, min=1, max=10, step=1,
                                 description='Runs/day:')
buff_cost_slider = widgets.FloatSlider(value=econ_params.buff_cost_scrap, min=0, max=200, step=10,
                                       description='Buff cost:')
bp_rate_slider = widgets.FloatSlider(value=econ_params.battle_pass.purchase_rate * 100,
                                      min=1, max=30, step=1, description='BP buyers %:')

# ── Defaults for reset/save ────────────────────────────────────────────────
_econ_defaults = {
    'coin_value': coin_value.value,
    'nut_value': nut_value.value,
    'scrap_value': scrap_value.value,
    'runs_per_day': runs_slider.value,
    'buff_cost': buff_cost_slider.value,
    'bp_rate': bp_rate_slider.value,
    'bronze_coin_cost': bronze_coin_cost.value,
    'merge_costs': {kc.name: _kc_merge_inputs[kc.name].value for kc in _kc_tiers},
    'cards_required': {kc.name: _kc_cards_inputs[kc.name].value for kc in _kc_tiers},
    'loot_nuts': {t.name: _loot_nuts[t.name].value for t in econ_params.instance_tiers},
    'loot_scrap': {t.name: _loot_scrap[t.name].value for t in econ_params.instance_tiers},
    'loot_coins': {t.name: _loot_coins[t.name].value for t in econ_params.instance_tiers},
    'loot_gear': {t.name: _loot_gear[t.name].value for t in econ_params.instance_tiers},
    'loot_kc_drop': {t.name: _loot_kc_drop[t.name].value for t in econ_params.instance_tiers},
}

def _reset_to_defaults(_):
    coin_value.value = _econ_defaults['coin_value']
    nut_value.value = _econ_defaults['nut_value']
    scrap_value.value = _econ_defaults['scrap_value']
    runs_slider.value = _econ_defaults['runs_per_day']
    buff_cost_slider.value = _econ_defaults['buff_cost']
    bp_rate_slider.value = _econ_defaults['bp_rate']
    bronze_coin_cost.value = _econ_defaults['bronze_coin_cost']
    for kc in _kc_tiers:
        _kc_merge_inputs[kc.name].value = _econ_defaults['merge_costs'][kc.name]
        _kc_cards_inputs[kc.name].value = _econ_defaults['cards_required'][kc.name]
    for t in econ_params.instance_tiers:
        _loot_nuts[t.name].value = _econ_defaults['loot_nuts'][t.name]
        _loot_scrap[t.name].value = _econ_defaults['loot_scrap'][t.name]
        _loot_coins[t.name].value = _econ_defaults['loot_coins'][t.name]
        _loot_gear[t.name].value = _econ_defaults['loot_gear'][t.name]
        _loot_kc_drop[t.name].value = _econ_defaults['loot_kc_drop'][t.name]
    save_status.value = 'Reset to defaults'

def _save_as_defaults(_):
    import yaml
    _econ_defaults['coin_value'] = coin_value.value
    _econ_defaults['nut_value'] = nut_value.value
    _econ_defaults['scrap_value'] = scrap_value.value
    _econ_defaults['runs_per_day'] = runs_slider.value
    _econ_defaults['buff_cost'] = buff_cost_slider.value
    _econ_defaults['bp_rate'] = bp_rate_slider.value
    _econ_defaults['bronze_coin_cost'] = bronze_coin_cost.value
    for kc in _kc_tiers:
        _econ_defaults['merge_costs'][kc.name] = _kc_merge_inputs[kc.name].value
        _econ_defaults['cards_required'][kc.name] = _kc_cards_inputs[kc.name].value
    for t in econ_params.instance_tiers:
        _econ_defaults['loot_nuts'][t.name] = _loot_nuts[t.name].value
        _econ_defaults['loot_scrap'][t.name] = _loot_scrap[t.name].value
        _econ_defaults['loot_coins'][t.name] = _loot_coins[t.name].value
        _econ_defaults['loot_gear'][t.name] = _loot_gear[t.name].value
        _econ_defaults['loot_kc_drop'][t.name] = _loot_kc_drop[t.name].value
    # Save economy params to config.yaml
    params = _make_econ_params(runs_slider.value, buff_cost_slider.value, bp_rate_slider.value)
    config_path = Path('config.yaml')
    with open(config_path) as f:
        data = yaml.safe_load(f) or {}
    data['economy'] = yaml.safe_load(params.model_dump_json())
    data['economy']['coin_to_usd'] = coin_value.value
    data['economy']['bronze_coin_cost'] = bronze_coin_cost.value
    data['economy']['nut_value_usd'] = nut_value.value
    data['economy']['scrap_value_usd'] = scrap_value.value
    with open(config_path, 'w') as f:
        yaml.dump(data, f, default_flow_style=None, sort_keys=False)
    save_status.value = f'Saved to config.yaml'

reset_btn = widgets.Button(description='Reset to Defaults', button_style='warning',
                           layout=widgets.Layout(width='160px'))
reset_btn.on_click(_reset_to_defaults)
save_btn = widgets.Button(description='Save as Defaults', button_style='success',
                          layout=widgets.Layout(width='160px'))
save_btn.on_click(_save_as_defaults)
save_status = widgets.Label(value='')

_anchors = state.retention_anchors if state else cfg.retention.anchors
_monetization = state.monetization if state else cfg.monetization

def _get_keycard_coin_costs():
    """Derive coin costs and total USD value (including merge costs) for each tier."""
    coin_costs = {}
    total_usd = {}  # full acquisition cost including cascading merges
    cv = coin_value.value
    nv = nut_value.value
    base = bronze_coin_cost.value
    coin_costs[_kc_names[0]] = base
    total_usd[_kc_names[0]] = base * cv  # bronze = just coin cost
    for i in range(1, len(_kc_tiers)):
        name = _kc_names[i]
        cards_needed = _kc_cards_inputs[name].value
        merge_nuts = _kc_merge_inputs[name].value
        prev_name = _kc_names[i - 1]
        # Coin cost = previous tier coin cost × cards needed
        prev_coin = coin_costs[prev_name]
        coin_costs[name] = prev_coin * cards_needed if cards_needed > 0 else prev_coin
        # Total USD = (cards × previous tier total USD) + merge nut cost in USD
        prev_total = total_usd[prev_name]
        total_usd[name] = (cards_needed * prev_total) + (merge_nuts * nv)
    return coin_costs, total_usd

def _make_econ_params(runs, buff_cost, bp_rate):
    from aco_model.models import KeyCardTier, InstanceTier
    updated_kc = []
    for kc in _kc_tiers:
        updated_kc.append(KeyCardTier(
            name=kc.name,
            cards_required=_kc_cards_inputs[kc.name].value,
            merge_cost_nuts=_kc_merge_inputs[kc.name].value,
            instance_tier=kc.instance_tier,
        ))
    updated_tiers = []
    for tier in econ_params.instance_tiers:
        updated_tiers.append(InstanceTier(
            name=tier.name,
            nuts_earned=_loot_nuts[tier.name].value,
            scrap_earned=_loot_scrap[tier.name].value,
            coins_earned=_loot_coins[tier.name].value,
            gear_value_usd=_loot_gear[tier.name].value,
            keycard_drop_chance=_loot_kc_drop[tier.name].value / 100.0,
        ))
    bp = econ_params.battle_pass.model_copy(update={'purchase_rate': bp_rate / 100.0})
    return econ_params.model_copy(update={
        'instances_per_day': runs, 'buff_cost_scrap': buff_cost,
        'battle_pass': bp, 'keycard_tiers': updated_kc,
        'instance_tiers': updated_tiers, 'coin_to_usd': coin_value.value,
    })

# ── Output widgets ────────────────────────────────────────────────────────
out_rates = widgets.Output()
out_ie = widgets.Output()
out_flows = widgets.Output()
out_wallet = widgets.Output()
out_kc = widgets.Output()
out_bp = widgets.Output()

def _update_all(*args):
    runs = runs_slider.value
    buff_cost = buff_cost_slider.value
    bp_rate = bp_rate_slider.value
    nv = nut_value.value
    sv = scrap_value.value
    cv = coin_value.value
    kc_costs, kc_total_usd = _get_keycard_coin_costs()

    params = _make_econ_params(runs, buff_cost, bp_rate)
    econ = simulate_economy(sim, params)
    days = np.arange(1, sim.sim_days + 1)

    # Update derived price labels
    for name, cost in kc_costs.items():
        usd = cost * cv
        _kc_derived_prices[name].value = f'<div style="width:120px;line-height:28px;font-family:monospace">{cost:,.2f}</div>'
        _kc_usd_labels[name].value = f'<div style="width:100px;line-height:28px;font-family:monospace">${kc_total_usd[name]:,.2f}</div>'

    # ── Exchange Rate & Value Tables ──────────────────────────────────
    with out_rates:
        out_rates.clear_output(wait=True)

        # Exchange rates (keycard row uses bronze cost as the base keycard value)
        base_kc_usd = kc_costs[_kc_names[0]] * cv
        rates_data = {
            'Resource': ['1 Coin', '1 Nut', '1 Scrap', f'1 Keycard (bronze)'],
            'USD': [f'${cv:.2f}', f'${nv:.4f}', f'${sv:.4f}', f'${base_kc_usd:.2f}'],
            'Coins': ['1.00',
                f'{nv / cv:.4f}' if cv > 0 else '—',
                f'{sv / cv:.4f}' if cv > 0 else '—',
                f'{base_kc_usd / cv:.2f}' if cv > 0 else '—'],
            'Nuts': [f'{cv / nv:,.0f}' if nv > 0 else '—', '1.00',
                f'{sv / nv:.2f}' if nv > 0 else '—',
                f'{base_kc_usd / nv:,.0f}' if nv > 0 else '—'],
            'Scrap': [f'{cv / sv:,.0f}' if sv > 0 else '—',
                f'{nv / sv:.2f}' if sv > 0 else '—', '1.00',
                f'{base_kc_usd / sv:,.0f}' if sv > 0 else '—'],
        }
        print('=== Exchange Rates ===')
        print(pd.DataFrame(rates_data).to_string(index=False))

        # Keycard cost summary
        print('\n=== Key Card Costs ===')
        kc_rows = []
        for i, kc in enumerate(params.keycard_tiers):
            coin_cost = kc_costs[kc.name]
            merge_nuts = kc.merge_cost_nuts
            merge_usd = merge_nuts * nv
            total_val = kc_total_usd[kc.name]
            prev_tier = _kc_names[i - 1] if i > 0 else '—'
            cards_label = f'{kc.cards_required} {prev_tier}' if kc.cards_required > 0 else '—'
            kc_rows.append({
                'Keycard': kc.name,
                'Buy (coins)': f'{coin_cost:,.0f}',
                'Total Value': f'${total_val:,.2f}',
                'Cards to Merge': cards_label,
                'Merge (nuts)': f'{merge_nuts:.0f}' if merge_nuts > 0 else '—',
                'Merge (USD)': f'${merge_usd:.2f}' if merge_nuts > 0 else '—',
                'Unlocks': kc.instance_tier,
            })
        print(pd.DataFrame(kc_rows).to_string(index=False))

        # Instance value per run in USD
        print('\n=== Instance Value per Run (USD) ===')
        rows = []
        for i, tier in enumerate(params.instance_tiers):
            kct = params.keycard_tiers[i] if i < len(params.keycard_tiers) else None
            kc_usd = kc_total_usd[kct.name] if kct else 0
            buff_usd = buff_cost * sv * params.buffs_per_run
            merge_usd = 0
            if kct and kct.merge_cost_nuts > 0 and kct.cards_required > 0:
                merge_usd = (kct.merge_cost_nuts * nv) / kct.cards_required
            total_in = buff_usd + kc_usd + merge_usd
            nuts_out_usd = tier.nuts_earned * nv
            scrap_out_usd = tier.scrap_earned * sv
            coins_out_usd = tier.coins_earned * cv
            gear_out_usd = tier.gear_value_usd
            keycard_back_usd = tier.keycard_drop_chance * kc_total_usd[kct.name] if kct else 0
            total_out = nuts_out_usd + scrap_out_usd + coins_out_usd + gear_out_usd + keycard_back_usd
            rows.append({
                'Tier': tier.name, 'KC In': f'${kc_usd:.2f}',
                'Buff': f'${buff_usd:.2f}', 'Merge': f'${merge_usd:.2f}',
                'Total In': f'${total_in:.2f}', 'Nuts': f'${nuts_out_usd:.2f}',
                'Scrap': f'${scrap_out_usd:.2f}', 'Coins': f'${coins_out_usd:.2f}',
                'Gear': f'${gear_out_usd:.2f}', 'KC Back': f'${keycard_back_usd:.2f}',
                'Total Out': f'${total_out:.2f}',
                'Net': f'${total_out - total_in:+.2f}',
            })
        print(pd.DataFrame(rows).to_string(index=False))
        avg_net = np.mean([float(r['Net'].replace('$','').replace('+','')) for r in rows])
        print(f'\nAvg net value per run: ${avg_net:+.2f} | Per player per day ({runs} runs): ${avg_net * runs:+.2f}')

    # ── Instance Economics (USD) ─────────────────────────────────────
    with out_ie:
        out_ie.clear_output(wait=True)
        # Build USD value arrays from the rows we already computed
        tier_names = [r['Tier'] for r in rows]
        total_in_vals = [float(r['Total In'].replace('$','')) for r in rows]
        total_out_vals = [float(r['Total Out'].replace('$','')) for r in rows]
        net_vals = [float(r['Net'].replace('$','').replace('+','')) for r in rows]

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        x = np.arange(len(tier_names)); width = 0.35
        ax1.bar(x - width/2, total_in_vals, width, label='Value In', color='#F44336', alpha=0.7)
        ax1.bar(x + width/2, total_out_vals, width, label='Value Out', color='#4CAF50', alpha=0.7)
        ax1.set_xticks(x); ax1.set_xticklabels(tier_names)
        ax1.set_ylabel('USD'); ax1.set_title('Value In vs Out per Instance Run (USD)'); ax1.legend()
        colors = ['#4CAF50' if v >= 0 else '#F44336' for v in net_vals]
        ax2.bar(tier_names, net_vals, color=colors, alpha=0.7)
        ax2.set_ylabel('USD'); ax2.set_title('Net Value per Instance Run (USD)')
        ax2.axhline(y=0, color='black', linestyle='-', alpha=0.3)
        plt.tight_layout(); plt.show()

    # ── Currency Flows ────────────────────────────────────────────────
    with out_flows:
        out_flows.clear_output(wait=True)
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        for ax, earned, spent, name in [
            (axes[0], econ.daily_nuts_earned, econ.daily_nuts_spent, 'Nuts'),
            (axes[1], econ.daily_scrap_earned, econ.daily_scrap_spent, 'Scrap'),
        ]:
            ax.fill_between(days, earned, alpha=0.3, color='#4CAF50', label='Earned')
            ax.fill_between(days, spent, alpha=0.3, color='#F44336', label='Spent')
            ax.plot(days, earned, linewidth=2, color='#4CAF50')
            ax.plot(days, spent, linewidth=2, color='#F44336')
            ax.set_xlabel('Day'); ax.set_ylabel(name); ax.set_title(f'Daily {name}: Earned vs Spent'); ax.legend()
        axes[2].plot(days, econ.daily_keycards_consumed, linewidth=2, color='#F44336', label='Consumed')
        axes[2].plot(days, econ.daily_keycards_net, linewidth=2, color='#FF9800', label='Net consumed')
        axes[2].set_xlabel('Day'); axes[2].set_ylabel('Keycards'); axes[2].set_title('Daily Keycard Consumption'); axes[2].legend()
        plt.tight_layout(); plt.show()

    # ── Wallet Balances ───────────────────────────────────────────────
    with out_wallet:
        out_wallet.clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        ax1.plot(days, econ.nuts_balance, linewidth=2, color='#FF9800', label='Nuts')
        ax1.plot(days, econ.scrap_balance, linewidth=2, color='#9C27B0', label='Scrap')
        ax1.set_xlabel('Day'); ax1.set_ylabel('Cumulative Balance'); ax1.set_title('Total Currency in System'); ax1.legend()
        dau = sim.dau.astype(float)
        with np.errstate(divide='ignore', invalid='ignore'):
            nuts_pp = np.where(dau > 0, econ.nuts_balance / dau, 0)
            scrap_pp = np.where(dau > 0, econ.scrap_balance / dau, 0)
        ax2.plot(days, nuts_pp, linewidth=2, color='#FF9800', label='Nuts/player')
        ax2.plot(days, scrap_pp, linewidth=2, color='#9C27B0', label='Scrap/player')
        ax2.set_xlabel('Day'); ax2.set_ylabel('Avg Balance per Player'); ax2.set_title('Avg Wallet Balance per Active Player'); ax2.legend()
        plt.tight_layout(); plt.show()
        df = econ.to_dataframe()
        print(f'Day {sim.sim_days}: Nuts balance = {df.iloc[-1]["nuts_balance"]:,} | Scrap balance = {df.iloc[-1]["scrap_balance"]:,}')

    # ── Key Card Progression ──────────────────────────────────────────
    with out_kc:
        out_kc.clear_output(wait=True)
        kc_df = econ.keycard_progression_dataframe()
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        ax1.bar(kc_df['tier'], kc_df['merge_cost_nuts'], color='#FF9800', alpha=0.7)
        ax1.set_ylabel('Nuts'); ax1.set_title('Merge Cost per Tier')
        ax2.bar(kc_df['tier'], kc_df['cumulative_nuts'], color='#F44336', alpha=0.7, label='Cumulative nuts')
        ax2_r = ax2.twinx()
        ax2_r.plot(kc_df['tier'], kc_df['cumulative_cards'], 'o-', color='#2196F3', linewidth=2, label='Cumulative cards')
        ax2.set_ylabel('Nuts', color='#F44336'); ax2_r.set_ylabel('Cards', color='#2196F3')
        ax2.set_title('Cumulative Investment to Reach Tier')
        ax2.legend(loc='upper left'); ax2_r.legend(loc='upper right')
        plt.tight_layout(); plt.show()
        avg_nuts_per_run = np.mean([t.nuts_earned for t in params.instance_tiers])
        nuts_per_day = avg_nuts_per_run * params.instances_per_day
        print(f'Avg nuts earned per day (per player): {nuts_per_day:,.0f}')
        for i, row in kc_df.iterrows():
            if row['cumulative_nuts'] > 0:
                print(f'  {row["tier"]}: ~{row["cumulative_nuts"] / nuts_per_day:.1f} days to afford (${row["cumulative_nuts"] * nv:.2f} USD)')

    # ── Battle Pass ───────────────────────────────────────────────────
    with out_bp:
        out_bp.clear_output(wait=True)
        bp = params.battle_pass
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        cum_bp_rev = np.cumsum(econ.battle_pass_daily_revenue)
        ax1.plot(days, cum_bp_rev, linewidth=2, color='#4CAF50')
        ax1.fill_between(days, cum_bp_rev, alpha=0.2, color='#4CAF50')
        ax1.set_xlabel('Day'); ax1.set_ylabel('Cumulative BP Revenue (USD)'); ax1.set_title('Battle Pass Revenue')
        rates_list = [0.1, 0.2, 0.3, 0.5, 0.7]
        for rate in rates_list:
            bp_var = bp.model_copy(update={'completion_rate': rate})
            params_var = params.model_copy(update={'battle_pass': bp_var})
            econ_var = simulate_economy(sim, params_var)
            net_coins = np.cumsum(econ_var.daily_coins_from_bp - econ_var.daily_coins_returned_bp)
            style = {'linewidth': 3, 'linestyle': '--'} if rate == bp.completion_rate else {'linewidth': 2}
            ax2.plot(days, net_coins * cv, label=f'{rate:.0%} complete', **style)
        ax2.set_xlabel('Day'); ax2.set_ylabel('Net Coin Revenue (USD)')
        ax2.set_title('BP Net Revenue by Completion Rate'); ax2.legend()
        plt.tight_layout(); plt.show()
        bp_nuts_usd = bp.nuts_reward_total * nv
        bp_scrap_usd = bp.scrap_reward_total * sv
        bp_kc_usd = bp.keycards_rewarded * kc_costs[_kc_names[0]] * cv
        bp_coins_usd = bp.coins_returned * cv
        bp_total_value = bp_nuts_usd + bp_scrap_usd + bp_kc_usd + bp_coins_usd
        bp_cost_usd = bp.cost_coins * cv
        print(f'BP cost: ${bp_cost_usd:.2f} | BP reward value: ${bp_total_value:.2f} (if completed)')
        print(f'  Coins back: ${bp_coins_usd:.2f} | Nuts: ${bp_nuts_usd:.2f} | Scrap: ${bp_scrap_usd:.2f} | Keycards: ${bp_kc_usd:.2f}')
        print(f'  Player ROI: {bp_total_value / bp_cost_usd:.1f}x (if completed)' if bp_cost_usd > 0 else '')
        print(f'Total BP revenue: ${econ.battle_pass_total_revenue:,.2f}')

    save_state(sim, _anchors, monetization=_monetization, economy=params)

# ── Build keycard input grid ──────────────────────────────────────────────
_kc_rows_ui = []

# Bronze row — editable coin cost
_kc_rows_ui.append(widgets.HBox([
    widgets.HTML('<div style="width:90px;line-height:28px;font-weight:bold">bronze</div>'),
    bronze_coin_cost,
    widgets.HTML('<div style="width:120px;line-height:28px;color:#999">(base tier)</div>'),
    widgets.HTML('<div style="width:120px"></div>'),
    widgets.HTML('<div style="width:100px"></div>'),
    _kc_usd_labels[_kc_names[0]],
]))

# Higher tiers — derived coin cost, editable merge
for i in range(1, len(_kc_tiers)):
    kc = _kc_tiers[i]
    prev_name = _kc_names[i - 1]
    _kc_rows_ui.append(widgets.HBox([
        widgets.HTML(f'<div style="width:90px;line-height:28px;font-weight:bold">{kc.name}</div>'),
        _kc_derived_prices[kc.name],
        _kc_merge_inputs[kc.name],
        _kc_cards_inputs[kc.name],
        widgets.HTML(f'<div style="width:100px;line-height:28px">{prev_name} cards</div>'),
        _kc_usd_labels[kc.name],
    ]))

_kc_header_row = widgets.HBox([
    widgets.HTML('<div style="width:90px;font-weight:bold">Tier</div>'),
    widgets.HTML('<div style="width:120px;font-weight:bold">Buy Price (coins)</div>'),
    widgets.HTML('<div style="width:120px;font-weight:bold">Merge Cost (nuts)</div>'),
    widgets.HTML('<div style="width:220px;font-weight:bold">Cards Required</div>'),
    widgets.HTML('<div style="width:100px;font-weight:bold">$ Value</div>'),
])


# ── Build loot input grid ─────────────────────────────────────────────────
_loot_header = widgets.HBox([
    widgets.HTML('<div style="width:90px;font-weight:bold">Tier</div>'),
    widgets.HTML('<div style="width:100px;font-weight:bold">Nuts</div>'),
    widgets.HTML('<div style="width:100px;font-weight:bold">Scrap</div>'),
    widgets.HTML('<div style="width:100px;font-weight:bold">Coins</div>'),
    widgets.HTML('<div style="width:100px;font-weight:bold">Gear ($)</div>'),
    widgets.HTML('<div style="width:100px;font-weight:bold">KC Drop %</div>'),
])
_loot_rows_ui = []
for tier in econ_params.instance_tiers:
    _loot_rows_ui.append(widgets.HBox([
        widgets.HTML(f'<div style="width:90px;line-height:28px;font-weight:bold">{tier.name}</div>'),
        _loot_nuts[tier.name],
        _loot_scrap[tier.name],
        _loot_coins[tier.name],
        _loot_gear[tier.name],
        _loot_kc_drop[tier.name],
    ]))
# ── Collect all observable widgets ────────────────────────────────────────
_all_inputs = [runs_slider, buff_cost_slider, bp_rate_slider, coin_value, nut_value, scrap_value, bronze_coin_cost]
for t in econ_params.instance_tiers:
    _all_inputs.extend([_loot_nuts[t.name], _loot_scrap[t.name], _loot_coins[t.name], _loot_gear[t.name], _loot_kc_drop[t.name]])
for kc in _kc_tiers:
    _all_inputs.extend([_kc_merge_inputs[kc.name], _kc_cards_inputs[kc.name]])

if _HEADLESS:
    _update_all()
else:
    for s in _all_inputs:
        s.observe(lambda change: _update_all(), names='value')

    display(widgets.VBox([
        widgets.HBox([coin_value, nut_value, scrap_value]),
        widgets.HBox([runs_slider, buff_cost_slider, bp_rate_slider]),
        widgets.HBox([reset_btn, save_btn, save_status]),
        widgets.HTML('<hr><b>Key Card Costs</b>'),
        _kc_header_row,
        *_kc_rows_ui,
        widgets.HTML('<hr><b>Instance Loot (avg per run)</b>'),
        _loot_header,
        *_loot_rows_ui,
        widgets.HTML('<hr>'),
        out_rates,
        out_ie,
    ]))
    _update_all()


## 2. Currency Flows

In [4]:
if not os.environ.get('ACO_HEADLESS'):
    display(out_flows)

Output(outputs=({'output_type': 'display_data', 'data': {'text/plain': '<Figure size 1800x500 with 3 Axes>', '…

## 3. Wallet Balances

Cumulative currency in the system over time (per GameGou call recommendations).

In [5]:
if not os.environ.get('ACO_HEADLESS'):
    display(out_wallet)

Output(outputs=({'output_type': 'display_data', 'data': {'text/plain': '<Figure size 1390x490 with 2 Axes>', '…

## 4. Key Card Progression

In [6]:
if not os.environ.get('ACO_HEADLESS'):
    display(out_kc)

Output(outputs=({'output_type': 'display_data', 'data': {'text/plain': '<Figure size 1400x500 with 3 Axes>', '…

## 5. Battle Pass Economics

In [7]:
if not os.environ.get('ACO_HEADLESS'):
    display(out_bp)

Output(outputs=({'output_type': 'display_data', 'data': {'text/plain': '<Figure size 1400x500 with 2 Axes>', '…